# QCxMS2 vs ICICLE Comparison

Compare ICICLE predicted spectra against experimental and QCxMS/QCxMS2 predictions.

QCxMS2 data: https://github.com/grimme-lab/QCxMS2-data

## Train-Set Contamination Check

Checks all NIST splits for whether QCxMS2 molecules appear in the train set.  
**Note:** 14/16 molecules are in NIST; no split keeps them out of train — treat ICICLE results as upper-bound / memorization for those molecules.

In [ ]:
MOLECULES = {
    "n-octane": "CCCCCCCC",
    "4-methyl-1-pentene": "CC(CC=C)C",
    "ethyl_propyl_ether": "CCOCCC",
    "1-butanol": "CCCCO",
    "butanal": "CCCC=O",
    "2-pentanone": "CCCC(C)=O",
    "butanoic_acid": "CCCC(O)=O",
    "methyl_butyrate": "CCCC(=O)OC",
    "butanamide": "CCCC(N)=O",
    "uracil": "O=C1NC(=O)NC=C1",
    "adenine": "Nc1ncnc2[nH]cnc12",
    "caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "tabun": "CCOP(=O)(C#N)N(C)C",
    "tetramethylbiphosphine_disulfide": "CP(=S)(C)P(=S)(C)C",
    "acibenzolar-S-methyl": "CSC(=O)c1ccc2scnc2c1",
    # "dichloroethylalumnium": "CC[Al](Cl)Cl",
}

QCXMS_THEORIES = [
    "gfn2_gfn2_qcxms2",
    "gfn2_qcxms",
    "wb97x3c_gfn2_qcxms2",
    "wb97x3c_wb97x3c_qcxms2",
]

N_BINS = 750

In [ ]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import MolToInchiKey
from IPython.display import display

_ik14s = {
    name: MolToInchiKey(Chem.MolFromSmiles(smi))[:14]
    for name, smi in MOLECULES.items()
}

_splits_dir = Path("/home/magled/icicle-dev/data/NIST2023_GCMS_main/splits")
_split_files = sorted(_splits_dir.glob("*.tsv"))

_records = []
for split_file in _split_files:
    df_s = pd.read_csv(split_file, sep="\t")
    ik_col = next(
        (c for c in ("inchi_key", "inchikey") if c in df_s.columns), None
    )
    if ik_col is None:
        continue  # random.tsv / random_no_xeno_aas.tsv have no inchi_key column
    df_s["ik14"] = df_s[ik_col].str[:14]
    all_ik14s = set(df_s["ik14"])
    for name, ik14 in _ik14s.items():
        label = (
            df_s[df_s["ik14"] == ik14]["split"].values[0]
            if ik14 in all_ik14s
            else "NOT_IN_NIST"
        )
        _records.append(
            {
                "split_file": split_file.name,
                "molecule": name,
                "split_label": label,
            }
        )

_contamination = pd.DataFrame(_records).pivot(
    index="molecule", columns="split_file", values="split_label"
)


# color cells: train=red, val/test=green, NOT_IN_NIST=gray
def _color(val):
    if val == "train":
        return "background-color: #f4a0a0"
    if val in ("val", "test"):
        return "background-color: #a0d8a0"
    return "background-color: #e0e0e0; color: #888"


print(
    "Red = train (memorized)  |  Green = val/test (unseen)  |  Gray = not in NIST"
)
display(_contamination.style.applymap(_color))

In [ ]:
QCXMS_DATA_PATH = "/home/magled/QCxMS2-data/"
# CKPT = "/home/magled/icicle-dev/checkpoints/06-20-50_icicle_rdm_no_xeno_best_so_far/checkpoints/best-model-val_loss=0.1473-epoch=98.ckpt"
CKPT = "/home/magled/icicle-dev/checkpoints/entropy_random_s1/checkpoints/best-model-val_loss=0.1183-epoch=48.ckpt"
DEVICE = "cuda:0"
MAX_NODES = 100
THRESHOLD = 0.01

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

from icicle.analysis.metrics import entropy_similarity, cosine_similarity
from icicle.models.eims_predictor import EIMSPredictorFromFullEnumeration
from icicle.utils.visualization import set_style, FIGSIZE
from icicle.utils.visualization.mass_spectra import plot_mirrored_spectra

set_style("manuscript")

In [ ]:
eims_predictor = EIMSPredictorFromFullEnumeration(
    min_mz=0, max_mz=N_BINS, bin_width=1.0
)
eims_predictor.load_from_checkpoint(CKPT)
print("Model loaded.")

In [ ]:
def load_csv_spectrum(path: Path, n_bins: int = N_BINS) -> np.ndarray | None:
    """Load mz/intensity CSV and bin into fixed-length array. Returns None if file missing."""
    if not path.exists():
        return None
    df = pd.read_csv(path, names=["mz", "intensity"])
    spec = np.zeros(n_bins)
    for mz, inten in zip(df["mz"], df["intensity"]):
        idx = int(round(mz))
        if 0 <= idx < n_bins:
            spec[idx] += inten
    if spec.max() > 0:
        spec /= spec.max()
    return spec


def predict_icicle(smiles: str) -> np.ndarray:
    """Run ICICLE inference and return max-normalized binned spectrum."""
    result = eims_predictor.predict_from_smiles(
        smiles=smiles, max_nodes=MAX_NODES, threshold=THRESHOLD, device=DEVICE
    )
    spec = result["intensities"]
    return spec / spec.max()

### LaTeX Table Helper

In [ ]:
from scipy import stats


def df_to_latex_table(
    df: pd.DataFrame,
    col_order: list[str],
    col_headers: list[str],
    metric_label: str,
    caption: str,
    label: str,
) -> str:
    """Render a per-molecule similarity table in the paper's QCxMS-table style.

    Bolds the row-wise max, appends a mean +/- SEM row, and formats molecule
    names with underscores replaced by spaces (matching tab:res-qcxms).
    """
    body = df[col_order].dropna(how="all")

    lines = []
    lines.append("% \\begin{landscape}")
    lines.append("\\begin{table}[t]")
    lines.append("\\centering")
    lines.append(f"\\caption{{{caption}}}")
    lines.append(f"\\label{{{label}}}")
    lines.append("\\resizebox{1.37\\textwidth}{!}")
    lines.append("{")
    lines.append("\\begin{tabular}{l|" + "l|" * (len(col_order) - 1) + "l}")

    header_cells = " & ".join(
        f"$\\mathbf{{sim_{{{metric_label},{h}}}}}$\\ " for h in col_headers
    )
    lines.append("\\textbf{Molecule} & " + header_cells + "\\\\ \\hline")

    for molecule, row in body.iterrows():
        vals = row[col_order].values.astype(float)
        best = np.nanargmax(vals)
        cells = []
        for i, v in enumerate(vals):
            s = f"{v:.3f}"
            cells.append(f"\\textbf{{{s}}}" if i == best else s)
        name = molecule.replace("_", " ")
        lines.append(name + " & " + " & ".join(cells) + " \\\\")

    lines.append("\\hline")
    means = body[col_order].mean().values
    sems = body[col_order].apply(lambda c: stats.sem(c.dropna())).values
    best_mean = np.nanargmax(means)
    mean_cells = []
    for i, (m, s) in enumerate(zip(means, sems)):
        cell = f"${m:.2f}\\pm{s:.2f}$"
        mean_cells.append(
            f"$\\mathbf{{{m:.2f}\\pm{s:.2f}}}$" if i == best_mean else cell
        )
    lines.append("\\textbf{Avg. sim.} & " + " & ".join(mean_cells) + " \\\\")

    lines.append("\\end{tabular}")
    lines.append("}")
    lines.append("\\end{table}")
    lines.append("% \\end{landscape}")
    return "\n".join(lines)


TABLE_COL_ORDER = [
    "gfn2_qcxms",
    "gfn2_gfn2_qcxms2",
    "wb97x3c_gfn2_qcxms2",
    "wb97x3c_wb97x3c_qcxms2",
    "neims_vs_exp",
    "massformer_vs_exp",
    "rassp_vs_exp",
    "icicle_vs_exp",
]
TABLE_COL_HEADERS = [
    "QCxMS-GFN2",
    "QCxMS2-GFN2",
    "QCxMS2-DFT",
    "QCxMS2-DFT2",
    "NEIMS",
    "MassFormer",
    "RASSP",
    "ICICLE",
]

## Per-Molecule: Mirror Plots + Entropy Similarities

For each molecule: ICICLE prediction vs experimental + each QCxMS2 theory.

In [ ]:
qcxms_path = Path(QCXMS_DATA_PATH)

rows = []

for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    pred_spec = predict_icicle(smiles)

    display(HTML(f"<hr><h3>{name}</h3><b>SMILES:</b> {smiles}"))

    # ICICLE vs experimental
    if exp_spec is not None:
        sim_icicle = entropy_similarity(pred_spec, exp_spec)
        display(
            HTML(
                f"<b>ICICLE vs Experimental</b> &nbsp; entropy_sim={sim_icicle:.3f}"
            )
        )
        fig = plot_mirrored_spectra(
            true_spec=exp_spec,
            pred_spec=pred_spec,
            true_label="Experimental",
            predicted_label="ICICLE",
            true_smiles=smiles,
            figsize=FIGSIZE["default"],
            title=f"entropy_sim={sim_icicle:.3f}",
        )
        plt.show()
    else:
        sim_icicle = float("nan")
        display(
            HTML("<i style='color:#888'>No experimental spectrum found.</i>")
        )

    row = {"molecule": name, "icicle_vs_exp": sim_icicle}

    # each QCxMS theory vs experimental
    for theory in QCXMS_THEORIES:
        qcxms_spec = load_csv_spectrum(mol_dir / f"{theory}.csv")
        if qcxms_spec is None or exp_spec is None:
            row[theory] = float("nan")
            continue
        sim_theory = entropy_similarity(qcxms_spec, exp_spec)
        row[theory] = sim_theory
        display(
            HTML(
                f"<b>{theory} vs Experimental</b> &nbsp; entropy_sim={sim_theory:.3f}"
            )
        )
        fig = plot_mirrored_spectra(
            true_spec=exp_spec,
            pred_spec=qcxms_spec,
            true_label="Experimental",
            predicted_label=theory,
            true_smiles=smiles,
            figsize=FIGSIZE["default"],
            title=f"entropy_sim={sim_theory:.3f}",
        )
        plt.show()

    rows.append(row)

## Summary: Entropy Similarity vs Experimental

In [ ]:
df = pd.DataFrame(rows).set_index("molecule")
df = df.round(3)

# mean row
df.loc["mean"] = df.mean()
df

In [ ]:
# show the same but with 1-value in every cell

df_distance_instead_of_similarity = df.copy()
for col in df.columns:
    if col == "molecule":
        continue
    df_distance_instead_of_similarity[col] = 1 - df[col]

df_distance_instead_of_similarity = df_distance_instead_of_similarity.round(3)

# remove mean row
df_distance_instead_of_similarity = df_distance_instead_of_similarity.drop(
    "mean"
)

df_distance_instead_of_similarity

In [ ]:
# order rows by molecular weight (lightest to heaviest)

from rdkit import Chem
from rdkit.Chem import Descriptors


def mol_weight(smiles: str) -> float:
    mol = Chem.MolFromSmiles(smiles)
    return Descriptors.MolWt(mol)


df_distance_instead_of_similarity["mol_weight"] = (
    df_distance_instead_of_similarity.index.map(
        lambda name: mol_weight(MOLECULES[name])
    )
)

df_distance_instead_of_similarity = (
    df_distance_instead_of_similarity.sort_values("mol_weight").drop(
        columns="mol_weight"
    )
)

df_distance_instead_of_similarity.loc["mean"] = (
    df_distance_instead_of_similarity.mean()
)

df_distance_instead_of_similarity

In [ ]:
df_plot = df.drop(index="mean")

col_labels = {
    "icicle_vs_exp": "ICICLE",
    "gfn2_gfn2_qcxms2": "GFN2/GFN2\nQCxMS2",
    "gfn2_qcxms": "GFN2\nQCxMS",
    "wb97x3c_gfn2_qcxms2": "wB97X-3c/GFN2\nQCxMS2",
    "wb97x3c_wb97x3c_qcxms2": "wB97X-3c\nQCxMS2",
}

from icicle.utils.visualization.eval_plots import (
    MODEL_PALETTE_INDICES,
    plot_boxplot_with_points,
)
from icicle.utils.visualization.style import get_palette, make_fig, save_fig

palette = get_palette()

# ICICLE/NEIMS/RASSP/MassFormer share the fixed MODEL_PALETTE_INDICES mapping
# (used across similarity/retrieval figures too). QCxMS-family colors: plain
# QCxMS = 11, GFN2-based QCxMS2 = 9, wB97X-3c (DFT) QCxMS2 = 10.
col_colors = {
    "icicle_vs_exp": palette[MODEL_PALETTE_INDICES["ICICLE"]],
    "gfn2_qcxms": palette[11],
    "gfn2_gfn2_qcxms2": palette[9],
    "wb97x3c_gfn2_qcxms2": palette[10],
    "wb97x3c_wb97x3c_qcxms2": palette[10],
}

data = {col_labels[c]: df_plot[c].dropna().values for c in col_labels}
colors = {col_labels[c]: col_colors[c] for c in col_labels}

fig = plot_boxplot_with_points(
    data, colors, xlabel="Entropy similarity vs exp"
)
save_fig(fig, "boxplot_entropy_similarity", output_dir=".")
plt.show()

# ── scatter: ICICLE vs wB97X-3c/wB97X-3c QCxMS2 ──────────────────────────────
df_scatter = df_plot[["icicle_vs_exp", "wb97x3c_wb97x3c_qcxms2"]].dropna()

fig, ax = make_fig("square")

ax.scatter(
    df_scatter["wb97x3c_wb97x3c_qcxms2"],
    df_scatter["icicle_vs_exp"],
    s=30,
    color=col_colors["icicle_vs_exp"],
    alpha=0.85,
    zorder=3,
)

lims = [
    min(df_scatter.min()) - 0.05,
    max(df_scatter.max()) + 0.05,
]
ax.plot(lims, lims, "k--", linewidth=1, alpha=0.5, label="y = x")

for mol, row in df_scatter.iterrows():
    ax.annotate(
        mol.replace("_", " "),
        xy=(row["wb97x3c_wb97x3c_qcxms2"], row["icicle_vs_exp"]),
        ha="left",
        va="bottom",
        xytext=(3, 2),
        textcoords="offset points",
    )

ax.set_xlabel("wB97X-3c QCxMS2\nentropy similarity vs exp")
ax.set_ylabel("ICICLE\nentropy similarity vs exp")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()

save_fig(fig, "scatter_icicle_vs_wb97x3c", output_dir=".")
plt.show()

## Summary: Cosine Similarity vs Experimental

In [ ]:
# Recompute cosine similarities for all molecules and QCxMS theories
cosine_rows = []

for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    pred_spec = predict_icicle(smiles)

    sim_icicle = (
        cosine_similarity(pred_spec, exp_spec)
        if exp_spec is not None
        else float("nan")
    )
    row = {"molecule": name, "icicle_vs_exp": sim_icicle}

    for theory in QCXMS_THEORIES:
        qcxms_spec = load_csv_spectrum(mol_dir / f"{theory}.csv")
        row[theory] = (
            cosine_similarity(qcxms_spec, exp_spec)
            if qcxms_spec is not None and exp_spec is not None
            else float("nan")
        )

    cosine_rows.append(row)

df_cosine = pd.DataFrame(cosine_rows).set_index("molecule")
df_cosine = df_cosine.round(3)
df_cosine.loc["mean"] = df_cosine.mean()
df_cosine

In [ ]:
df_cosine_plot = df_cosine.drop(index="mean")

data_cosine = {
    col_labels[c]: df_cosine_plot[c].dropna().values for c in col_labels
}
colors_cosine = {col_labels[c]: col_colors[c] for c in col_labels}

fig = plot_boxplot_with_points(
    data_cosine, colors_cosine, xlabel="Cosine similarity vs exp"
)
save_fig(fig, "boxplot_cosine_similarity", output_dir=".")
plt.show()

## NEIMS Comparison

Load trained NEIMS checkpoint and predict spectra for the same QCxMS2 molecules.

In [ ]:
import sys
import json
import torch
import numpy as np
from pathlib import Path
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

NEIMS_DIR = Path("/home/magled/icicle-dev/baselines/neims")
NEIMS_CKPT = NEIMS_DIR / "outputs/neims_random_noqcxms2_s1/best_model.pt"

sys.path.insert(0, str(NEIMS_DIR))
from neims.model import NEIMS as NEIMSModel

checkpoint = torch.load(NEIMS_CKPT, map_location="cpu")
cfg = checkpoint["model_config"]

neims_model = NEIMSModel(
    input_size=cfg.get("fp_length", 4096),
    output_size=cfg["output_size"],
    hidden_sizes=cfg.get("hidden_sizes", [2000] * 8),
    dropout=cfg.get("dropout", 0.25),
    bidirectional=cfg.get("bidirectional", True),
    gate_bidirectional=cfg.get("gate_bidirectional", False),
    resnet_bottleneck=cfg.get("resnet_bottleneck", 0.5),
    max_mass_offset=cfg.get("max_mass_offset", 5),
    fp_radius=cfg.get("fp_radius", 2),
    fp_length=cfg.get("fp_length", 4096),
)
neims_model.load_state_dict(checkpoint["model_state_dict"])
neims_model.eval()

NEIMS_MIN_MZ = cfg.get("min_mz", 0.0)
NEIMS_MAX_MZ = cfg.get("max_mz", 750.0)
NEIMS_BIN_WIDTH = cfg.get("bin_width", 1.0)
fp_radius = cfg.get("fp_radius", 2)
fp_length = cfg.get("fp_length", 4096)

mfpgen = rdFingerprintGenerator.GetMorganGenerator(
    radius=fp_radius, fpSize=fp_length
)

print(
    f"NEIMS loaded. output_size={cfg['output_size']}, bidirectional={cfg.get('bidirectional')}"
)

In [ ]:
def predict_neims(smiles: str) -> np.ndarray:
    """Run NEIMS inference for a single SMILES. Returns max-normalized binned spectrum."""
    mol = Chem.MolFromSmiles(smiles)
    fp = mfpgen.GetCountFingerprintAsNumPy(mol).astype(np.float32)
    mass = np.float32(Descriptors.ExactMolWt(mol))

    with torch.no_grad():
        fp_t = torch.from_numpy(fp).unsqueeze(0)
        mass_t = torch.tensor(
            [[mass]]
        )  # shape [batch_size, 1] as expected by model
        pred = neims_model(fp_t, mass_t).squeeze(0).numpy()

    pred = np.maximum(pred, 0)
    if pred.max() > 0:
        pred /= pred.max()
    return pred


# Collect NEIMS similarities vs experimental
neims_rows = []

for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    neims_spec = predict_neims(smiles)

    sim_neims = (
        entropy_similarity(neims_spec, exp_spec)
        if exp_spec is not None
        else float("nan")
    )
    neims_rows.append({"molecule": name, "neims_vs_exp": sim_neims})

df_neims = pd.DataFrame(neims_rows).set_index("molecule")
df_neims = df_neims.round(3)
df_neims.loc["mean"] = df_neims.mean()
df_neims

In [ ]:
# Merge NEIMS with existing results and plot combined boxplot
df_combined = df_plot.join(df_neims.drop(index="mean"), how="left")

col_labels_combined = {
    "icicle_vs_exp": "ICICLE",
    "neims_vs_exp": "NEIMS",
    "gfn2_gfn2_qcxms2": "GFN2/GFN2\nQCxMS2",
    "gfn2_qcxms": "GFN2\nQCxMS",
    "wb97x3c_gfn2_qcxms2": "wB97X-3c/GFN2\nQCxMS2",
    "wb97x3c_wb97x3c_qcxms2": "wB97X-3c\nQCxMS2",
}

col_colors_combined = {
    "icicle_vs_exp": palette[MODEL_PALETTE_INDICES["ICICLE"]],
    "neims_vs_exp": palette[MODEL_PALETTE_INDICES["NEIMS"]],
    "gfn2_qcxms": palette[11],
    "gfn2_gfn2_qcxms2": palette[9],
    "wb97x3c_gfn2_qcxms2": palette[10],
    "wb97x3c_wb97x3c_qcxms2": palette[10],
}

data_combined = {
    col_labels_combined[c]: df_combined[c].dropna().values
    for c in col_labels_combined
}
colors_combined = {
    col_labels_combined[c]: col_colors_combined[c] for c in col_labels_combined
}

fig = plot_boxplot_with_points(
    data_combined, colors_combined, xlabel="Entropy similarity vs exp"
)
save_fig(fig, "boxplot_entropy_similarity_with_neims", output_dir=".")
plt.show()

In [ ]:
# Cosine similarity for NEIMS vs experimental
neims_cosine_rows = []

for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    neims_spec = predict_neims(smiles)

    sim_neims = (
        cosine_similarity(neims_spec, exp_spec)
        if exp_spec is not None
        else float("nan")
    )
    neims_cosine_rows.append({"molecule": name, "neims_vs_exp": sim_neims})

df_neims_cosine = pd.DataFrame(neims_cosine_rows).set_index("molecule")
df_neims_cosine = df_neims_cosine.round(3)
df_neims_cosine.loc["mean"] = df_neims_cosine.mean()

# Combined cosine boxplot (ICICLE + NEIMS + QCxMS theories)
df_combined_cosine = df_cosine_plot.join(
    df_neims_cosine.drop(index="mean"), how="left"
)

data_combined_cosine = {
    col_labels_combined[c]: df_combined_cosine[c].dropna().values
    for c in col_labels_combined
}

fig = plot_boxplot_with_points(
    data_combined_cosine, colors_combined, xlabel="Cosine similarity vs exp"
)
save_fig(fig, "boxplot_cosine_similarity_with_neims", output_dir=".")
plt.show()

In [ ]:
# Weighted cosine similarity for NEIMS vs experimental
neims_cosine_rows = []
from icicle.analysis.metrics import weighted_cosine_similarity

for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    neims_spec = predict_neims(smiles)

    sim_neims = (
        weighted_cosine_similarity(
            neims_spec, exp_spec, mz_values=np.arange(len(neims_spec))
        )
        if exp_spec is not None
        else float("nan")
    )
    neims_cosine_rows.append({"molecule": name, "neims_vs_exp": sim_neims})

df_neims_cosine = pd.DataFrame(neims_cosine_rows).set_index("molecule")
df_neims_cosine = df_neims_cosine.round(3)
df_neims_cosine.loc["mean"] = df_neims_cosine.mean()

# Combined weighted-cosine boxplot (ICICLE + NEIMS + QCxMS theories)
df_combined_cosine = df_cosine_plot.join(
    df_neims_cosine.drop(index="mean"), how="left"
)

data_combined_cosine = {
    col_labels_combined[c]: df_combined_cosine[c].dropna().values
    for c in col_labels_combined
}

fig = plot_boxplot_with_points(
    data_combined_cosine,
    colors_combined,
    xlabel="Weighted cosine similarity vs exp",
)
save_fig(fig, "boxplot_weighted_cosine_similarity_with_neims", output_dir=".")
plt.show()

## MassFormer Comparison

Load trained MassFormer checkpoint and predict spectra for the same QCxMS2 molecules.

In [ ]:
import subprocess
import json
import numpy as np

MF_PYTHON = "/home/magled/miniconda3/envs/MF-GPU/bin/python"
MF_SCRIPT = "/home/magled/icicle-dev/baselines/massformer/scripts/predict_smiles_to_json.py"
MF_CKPT = "/home/magled/icicle-dev/baselines/massformer/checkpoints/massformer_random_s2/chkpt.pkl"


def predict_massformer_batch(
    smiles_list: list[str], device: str = "cuda:0"
) -> dict[str, np.ndarray]:
    """Run MassFormer in MF-GPU env via subprocess. Returns {smiles: spectrum array}."""
    result = subprocess.run(
        [
            MF_PYTHON,
            MF_SCRIPT,
            "--smiles",
            *smiles_list,
            "--checkpoint",
            MF_CKPT,
            "--device",
            device,
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    if not result.stdout.strip():
        raise RuntimeError(
            f"MassFormer subprocess produced no output.\nstderr:\n{result.stderr}"
        )
    return {smi: np.array(v) for smi, v in json.loads(result.stdout).items()}


print("MassFormer subprocess helper ready.")

In [ ]:
# Run all SMILES in one subprocess call
mf_spectra = predict_massformer_batch(list(MOLECULES.values()))

mf_rows = []
for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    mf_spec = mf_spectra[smiles]  # already 750 bins, matches N_BINS
    sim_mf = (
        entropy_similarity(mf_spec, exp_spec)
        if exp_spec is not None
        else float("nan")
    )
    mf_rows.append({"molecule": name, "massformer_vs_exp": sim_mf})

df_mf = pd.DataFrame(mf_rows).set_index("molecule")
df_mf = df_mf.round(3)
df_mf.loc["mean"] = df_mf.mean()
df_mf

In [ ]:
for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    mf_spec = mf_spectra[smiles][:N_BINS]

    display(HTML(f"<hr><h3>{name}</h3><b>SMILES:</b> {smiles}"))

    if exp_spec is not None:
        sim_mf = entropy_similarity(mf_spec, exp_spec)
        display(
            HTML(
                f"<b>MassFormer vs Experimental</b> &nbsp; entropy_sim={sim_mf:.3f}"
            )
        )
        fig = plot_mirrored_spectra(
            true_spec=exp_spec,
            pred_spec=mf_spec,
            true_label="Experimental",
            predicted_label="MassFormer",
            true_smiles=smiles,
            figsize=FIGSIZE["default"],
            title=f"entropy_sim={sim_mf:.3f}",
        )
        plt.show()
    else:
        display(
            HTML("<i style='color:#888'>No experimental spectrum found.</i>")
        )

## RASSP Comparison

Load trained RASSP checkpoint (random split, seed 1) and predict spectra for the same QCxMS2 molecules.

In [ ]:
import subprocess
import json
import numpy as np

RASSP_PYTHON = "/home/magled/miniconda3/envs/rassp/bin/python"
RASSP_SCRIPT = (
    "/home/magled/icicle-dev/baselines/rassp/scripts/predict_smiles_to_json.py"
)
RASSP_CKPT = "/home/magled/icicle-dev/baselines/rassp/rassp/checkpoints/rassp_random_s1.rassp_random_s1.00000050.model"
RASSP_META = "/home/magled/icicle-dev/baselines/rassp/rassp/checkpoints/rassp_random_s1.rassp_random_s1.meta"


def predict_rassp_batch(
    smiles_list: list[str], device: str = "cuda:1", batch_size: int = 8
) -> dict[str, np.ndarray]:
    """Run RASSP in its own conda env via subprocess. Returns {smiles: spectrum array}.

    Uses a different GPU than ICICLE's (cuda:0) by default -- RASSP's own model
    is memory-heavy per molecule and easily OOMs when sharing a GPU already
    holding the ICICLE checkpoint loaded earlier in this notebook.
    """
    result = subprocess.run(
        [
            RASSP_PYTHON,
            RASSP_SCRIPT,
            "--smiles",
            *smiles_list,
            "--checkpoint",
            RASSP_CKPT,
            "--meta",
            RASSP_META,
            "--device",
            device,
            "--batch-size",
            str(batch_size),
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    if not result.stdout.strip():
        raise RuntimeError(
            f"RASSP subprocess produced no output.\nstderr:\n{result.stderr}"
        )
    return {smi: np.array(v) for smi, v in json.loads(result.stdout).items()}


print("RASSP subprocess helper ready.")

In [ ]:
# Run all SMILES in one subprocess call
rassp_spectra = predict_rassp_batch(list(MOLECULES.values()))

rassp_rows = []
for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    rassp_spec = rassp_spectra[smiles][:N_BINS]
    sim_rassp = (
        entropy_similarity(rassp_spec, exp_spec)
        if exp_spec is not None
        else float("nan")
    )
    rassp_rows.append({"molecule": name, "rassp_vs_exp": sim_rassp})

df_rassp = pd.DataFrame(rassp_rows).set_index("molecule")
df_rassp = df_rassp.round(3)
df_rassp.loc["mean"] = df_rassp.mean()
df_rassp

In [ ]:
for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    rassp_spec = rassp_spectra[smiles][:N_BINS]

    display(HTML(f"<hr><h3>{name}</h3><b>SMILES:</b> {smiles}"))

    if exp_spec is not None:
        sim_rassp = entropy_similarity(rassp_spec, exp_spec)
        display(
            HTML(
                f"<b>RASSP vs Experimental</b> &nbsp; entropy_sim={sim_rassp:.3f}"
            )
        )
        fig = plot_mirrored_spectra(
            true_spec=exp_spec,
            pred_spec=rassp_spec,
            true_label="Experimental",
            predicted_label="RASSP",
            true_smiles=smiles,
            figsize=FIGSIZE["default"],
            title=f"entropy_sim={sim_rassp:.3f}",
        )
        plt.show()
    else:
        display(
            HTML("<i style='color:#888'>No experimental spectrum found.</i>")
        )

In [ ]:
# Merge ICICLE + NEIMS + MassFormer + RASSP + QCxMS
df_neims_ent = pd.DataFrame(
    [
        {"molecule": r["molecule"], "neims_vs_exp": r["neims_vs_exp"]}
        for r in neims_rows
    ]
).set_index("molecule")

df_all = (
    df_plot.join(df_neims_ent, how="left")
    .join(df_mf.drop(index="mean"), how="left")
    .join(df_rassp.drop(index="mean"), how="left")
)

col_labels_all = {
    "icicle_vs_exp": "ICICLE",
    "neims_vs_exp": "NEIMS",
    "massformer_vs_exp": "MassFormer",
    "rassp_vs_exp": "RASSP",
    "gfn2_gfn2_qcxms2": "GFN2/GFN2\nQCxMS2",
    "gfn2_qcxms": "GFN2\nQCxMS",
    "wb97x3c_gfn2_qcxms2": "wB97X-3c/GFN2\nQCxMS2",
    "wb97x3c_wb97x3c_qcxms2": "wB97X-3c\nQCxMS2",
}

col_colors_all = {
    "icicle_vs_exp": palette[MODEL_PALETTE_INDICES["ICICLE"]],
    "neims_vs_exp": palette[MODEL_PALETTE_INDICES["NEIMS"]],
    "massformer_vs_exp": palette[MODEL_PALETTE_INDICES["MassFormer"]],
    "rassp_vs_exp": palette[MODEL_PALETTE_INDICES["RASSP"]],
    "gfn2_qcxms": palette[11],
    "gfn2_gfn2_qcxms2": palette[9],
    "wb97x3c_gfn2_qcxms2": palette[10],
    "wb97x3c_wb97x3c_qcxms2": palette[10],
}

data_all = {
    col_labels_all[c]: df_all[c].dropna().values for c in col_labels_all
}
colors_all = {col_labels_all[c]: col_colors_all[c] for c in col_labels_all}

fig = plot_boxplot_with_points(
    data_all, colors_all, xlabel="Entropy similarity"
)
save_fig(
    fig,
    "boxplot_entropy_similarity_with_neims_massformer_rassp",
    output_dir=".",
)
plt.show()

### Table: Entropy Similarity

In [ ]:
latex_table_entropy_similarity = df_to_latex_table(
    df_all,
    TABLE_COL_ORDER,
    TABLE_COL_HEADERS,
    metric_label="entr",
    caption="\\textbf{Performance of QCxMS and QCxMS2 compared to ICICLE, NEIMS, MassFormer, and RASSP (entropy similarity).}",
    label="tab:res-qcxms-table_entropy_similarity",
)
with open("table_entropy_similarity.tex", "w") as f:
    f.write(latex_table_entropy_similarity)
print(latex_table_entropy_similarity)

## Summary: Cosine Similarity vs Experimental (ICICLE + NEIMS + MassFormer + RASSP)

In [ ]:
# Cosine similarity: NEIMS/MassFormer/RASSP vs experimental (ICICLE already in df_cosine)
mf_cosine_rows = []
rassp_cosine_rows = []

for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")

    mf_spec = mf_spectra[smiles][:N_BINS]
    sim_mf = (
        cosine_similarity(mf_spec, exp_spec)
        if exp_spec is not None
        else float("nan")
    )
    mf_cosine_rows.append({"molecule": name, "massformer_vs_exp": sim_mf})

    rassp_spec = rassp_spectra[smiles][:N_BINS]
    sim_rassp = (
        cosine_similarity(rassp_spec, exp_spec)
        if exp_spec is not None
        else float("nan")
    )
    rassp_cosine_rows.append({"molecule": name, "rassp_vs_exp": sim_rassp})

df_mf_cosine = pd.DataFrame(mf_cosine_rows).set_index("molecule").round(3)
df_mf_cosine.loc["mean"] = df_mf_cosine.mean()

df_rassp_cosine = (
    pd.DataFrame(rassp_cosine_rows).set_index("molecule").round(3)
)
df_rassp_cosine.loc["mean"] = df_rassp_cosine.mean()

In [ ]:
# Merge ICICLE + NEIMS + MassFormer + RASSP + QCxMS cosine similarities
df_neims_cos = pd.DataFrame(
    [
        {"molecule": m, "neims_vs_exp": v}
        for m, v in df_neims_cosine.drop(index="mean")["neims_vs_exp"].items()
    ]
).set_index("molecule")

df_all_cosine = (
    df_cosine_plot.join(df_neims_cos, how="left")
    .join(df_mf_cosine.drop(index="mean"), how="left")
    .join(df_rassp_cosine.drop(index="mean"), how="left")
)

col_labels_all_cosine = {
    "icicle_vs_exp": "ICICLE",
    "neims_vs_exp": "NEIMS",
    "massformer_vs_exp": "MassFormer",
    "rassp_vs_exp": "RASSP",
    "gfn2_gfn2_qcxms2": "GFN2/GFN2\nQCxMS2",
    "gfn2_qcxms": "GFN2\nQCxMS",
    "wb97x3c_gfn2_qcxms2": "wB97X-3c/GFN2\nQCxMS2",
    "wb97x3c_wb97x3c_qcxms2": "wB97X-3c\nQCxMS2",
}

col_colors_all_cosine = {
    "icicle_vs_exp": palette[MODEL_PALETTE_INDICES["ICICLE"]],
    "neims_vs_exp": palette[MODEL_PALETTE_INDICES["NEIMS"]],
    "massformer_vs_exp": palette[MODEL_PALETTE_INDICES["MassFormer"]],
    "rassp_vs_exp": palette[MODEL_PALETTE_INDICES["RASSP"]],
    "gfn2_qcxms": palette[11],
    "gfn2_gfn2_qcxms2": palette[9],
    "wb97x3c_gfn2_qcxms2": palette[10],
    "wb97x3c_wb97x3c_qcxms2": palette[10],
}

data_all_cosine = {
    col_labels_all_cosine[c]: df_all_cosine[c].dropna().values
    for c in col_labels_all_cosine
}
colors_all_cosine = {
    col_labels_all_cosine[c]: col_colors_all_cosine[c]
    for c in col_labels_all_cosine
}

fig = plot_boxplot_with_points(
    data_all_cosine, colors_all_cosine, xlabel="Cosine similarity"
)
save_fig(
    fig,
    "boxplot_cosine_similarity_with_neims_massformer_rassp",
    output_dir=".",
)
plt.show()

### Table: Cosine Similarity

In [ ]:
latex_table_cosine_similarity = df_to_latex_table(
    df_all_cosine,
    TABLE_COL_ORDER,
    TABLE_COL_HEADERS,
    metric_label="cos",
    caption="\\textbf{Performance of QCxMS and QCxMS2 compared to ICICLE, NEIMS, MassFormer, and RASSP (cosine similarity).}",
    label="tab:res-qcxms-table_cosine_similarity",
)
with open("table_cosine_similarity.tex", "w") as f:
    f.write(latex_table_cosine_similarity)
print(latex_table_cosine_similarity)

## Summary: Weighted Cosine Similarity vs Experimental (ICICLE + NEIMS + MassFormer + RASSP + QCxMS)

In [ ]:
# Weighted Cosine Similarity: all models + QCxMS theories vs experimental
rows_metric = []

for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    mz_values = np.arange(N_BINS)

    def sim(pred_spec):
        if pred_spec is None or exp_spec is None:
            return float("nan")
        return weighted_cosine_similarity(
            pred_spec,
            exp_spec,
            mz_values=mz_values,
            weighting_scheme="nist_gc",
        )

    icicle_spec = predict_icicle(smiles)
    neims_spec = predict_neims(smiles)
    mf_spec = mf_spectra[smiles][:N_BINS]
    rassp_spec = rassp_spectra[smiles][:N_BINS]

    row = {
        "molecule": name,
        "icicle_vs_exp": sim(icicle_spec),
        "neims_vs_exp": sim(neims_spec),
        "massformer_vs_exp": sim(mf_spec),
        "rassp_vs_exp": sim(rassp_spec),
    }
    for theory in QCXMS_THEORIES:
        qcxms_spec = load_csv_spectrum(mol_dir / f"{theory}.csv")
        row[theory] = sim(qcxms_spec)

    rows_metric.append(row)

df_metric = pd.DataFrame(rows_metric).set_index("molecule").round(3)
df_metric

In [ ]:
col_labels_all_cosine = {
    "icicle_vs_exp": "ICICLE",
    "neims_vs_exp": "NEIMS",
    "massformer_vs_exp": "MassFormer",
    "rassp_vs_exp": "RASSP",
    "gfn2_gfn2_qcxms2": "GFN2/GFN2\nQCxMS2",
    "gfn2_qcxms": "GFN2\nQCxMS",
    "wb97x3c_gfn2_qcxms2": "wB97X-3c/GFN2\nQCxMS2",
    "wb97x3c_wb97x3c_qcxms2": "wB97X-3c\nQCxMS2",
}

col_colors_all_cosine = {
    "icicle_vs_exp": palette[MODEL_PALETTE_INDICES["ICICLE"]],
    "neims_vs_exp": palette[MODEL_PALETTE_INDICES["NEIMS"]],
    "massformer_vs_exp": palette[MODEL_PALETTE_INDICES["MassFormer"]],
    "rassp_vs_exp": palette[MODEL_PALETTE_INDICES["RASSP"]],
    "gfn2_qcxms": palette[11],
    "gfn2_gfn2_qcxms2": palette[9],
    "wb97x3c_gfn2_qcxms2": palette[10],
    "wb97x3c_wb97x3c_qcxms2": palette[10],
}

data_metric = {
    col_labels_all_cosine[c]: df_metric[c].dropna().values
    for c in col_labels_all_cosine
}
colors_metric = {
    col_labels_all_cosine[c]: col_colors_all_cosine[c]
    for c in col_labels_all_cosine
}

fig = plot_boxplot_with_points(
    data_metric, colors_metric, xlabel="Weighted cosine similarity"
)
save_fig(
    fig,
    "boxplot_weighted_cosine_similarity_with_neims_massformer_rassp",
    output_dir=".",
)
plt.show()

### Table: Weighted Cosine Similarity (NIST-GC)

In [ ]:
latex_table_weighted_cosine_similarity = df_to_latex_table(
    df_metric,
    TABLE_COL_ORDER,
    TABLE_COL_HEADERS,
    metric_label="wcos",
    caption="\\textbf{Performance of QCxMS and QCxMS2 compared to ICICLE, NEIMS, MassFormer, and RASSP (NIST-GC weighted cosine similarity).}",
    label="tab:res-qcxms-table_weighted_cosine_similarity",
)
with open("table_weighted_cosine_similarity.tex", "w") as f:
    f.write(latex_table_weighted_cosine_similarity)
print(latex_table_weighted_cosine_similarity)

## Summary: Composite Weighted Cosine Similarity vs Experimental (ICICLE + NEIMS + MassFormer + RASSP + QCxMS)

In [ ]:
from icicle.analysis.metrics import composite_weighted_cosine_similarity

# Composite Weighted Cosine Similarity: all models + QCxMS theories vs experimental
rows_metric = []

for name, smiles in MOLECULES.items():
    mol_dir = qcxms_path / name
    exp_spec = load_csv_spectrum(mol_dir / "exp.csv")
    mz_values = np.arange(N_BINS)

    def sim(pred_spec):
        if pred_spec is None or exp_spec is None:
            return float("nan")
        return composite_weighted_cosine_similarity(
            pred_spec,
            exp_spec,
            mz_values=mz_values,
            weighting_scheme="nist_gc",
        )

    icicle_spec = predict_icicle(smiles)
    neims_spec = predict_neims(smiles)
    mf_spec = mf_spectra[smiles][:N_BINS]
    rassp_spec = rassp_spectra[smiles][:N_BINS]

    row = {
        "molecule": name,
        "icicle_vs_exp": sim(icicle_spec),
        "neims_vs_exp": sim(neims_spec),
        "massformer_vs_exp": sim(mf_spec),
        "rassp_vs_exp": sim(rassp_spec),
    }
    for theory in QCXMS_THEORIES:
        qcxms_spec = load_csv_spectrum(mol_dir / f"{theory}.csv")
        row[theory] = sim(qcxms_spec)

    rows_metric.append(row)

df_metric = pd.DataFrame(rows_metric).set_index("molecule").round(3)
df_metric

In [ ]:
col_labels_all_cosine = {
    "icicle_vs_exp": "ICICLE",
    "neims_vs_exp": "NEIMS",
    "massformer_vs_exp": "MassFormer",
    "rassp_vs_exp": "RASSP",
    "gfn2_gfn2_qcxms2": "GFN2/GFN2\nQCxMS2",
    "gfn2_qcxms": "GFN2\nQCxMS",
    "wb97x3c_gfn2_qcxms2": "wB97X-3c/GFN2\nQCxMS2",
    "wb97x3c_wb97x3c_qcxms2": "wB97X-3c\nQCxMS2",
}

col_colors_all_cosine = {
    "icicle_vs_exp": palette[MODEL_PALETTE_INDICES["ICICLE"]],
    "neims_vs_exp": palette[MODEL_PALETTE_INDICES["NEIMS"]],
    "massformer_vs_exp": palette[MODEL_PALETTE_INDICES["MassFormer"]],
    "rassp_vs_exp": palette[MODEL_PALETTE_INDICES["RASSP"]],
    "gfn2_qcxms": palette[11],
    "gfn2_gfn2_qcxms2": palette[9],
    "wb97x3c_gfn2_qcxms2": palette[10],
    "wb97x3c_wb97x3c_qcxms2": palette[10],
}

data_metric = {
    col_labels_all_cosine[c]: df_metric[c].dropna().values
    for c in col_labels_all_cosine
}
colors_metric = {
    col_labels_all_cosine[c]: col_colors_all_cosine[c]
    for c in col_labels_all_cosine
}

fig = plot_boxplot_with_points(
    data_metric, colors_metric, xlabel="Composite similarity"
)
save_fig(
    fig,
    "boxplot_composite_similarity_with_neims_massformer_rassp",
    output_dir=".",
)
plt.show()

### Table: Composite Similarity

In [ ]:
latex_table_composite_similarity = df_to_latex_table(
    df_metric,
    TABLE_COL_ORDER,
    TABLE_COL_HEADERS,
    metric_label="comp",
    caption="\\textbf{Performance of QCxMS and QCxMS2 compared to ICICLE, NEIMS, MassFormer, and RASSP (composite weighted cosine similarity).}",
    label="tab:res-qcxms-table_composite_similarity",
)
with open("table_composite_similarity.tex", "w") as f:
    f.write(latex_table_composite_similarity)
print(latex_table_composite_similarity)